In [0]:
# =============================================================
# NOTEBOOK:  08_bronze_bad_data_handling
# PURPOSE:   Test corrupt record handling in Auto Loader
# SOURCE:    Intentionally corrupted CSV files
# TARGET:    Bronze Delta + Quarantine table
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid

# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("=" * 60)
print("RetailMax - Bronze: Bad Data Handling")
print("=" * 60)

# ── Paths ─────────────────────────────────────────────────────
source_path     = bronze_path + "raw_ingest/test/bad_data/"
target_path     = bronze_path + "tables/test/bad_data_customers/"
checkpoint_path = bronze_path + "_checkpoints/test/bad_data_customers/"
schema_path     = bronze_path + "_schema/test/bad_data_customers/"
quarantine_path = bronze_path + "_quarantine/test/bad_data_customers/"

# ── Generate Intentionally Bad CSV Data ────────────────────────
# Mix of good records and different types of corruption

csv_data = """customer_id,first_name,last_name,email,age,loyalty_points,is_active,segment
1,John,Smith,john@email.com,35,1500,1,Premium
2,Jane,Doe,jane@email.com,28,2500,1,VIP
3,Bob,Wilson,bob@email.com,45,800,0,Basic
4,Alice,Brown,alice@email.com,32,3200,1,Standard
5,Charlie,Davis,charlie@email.com,55,4100,1,Premium
6,Wrong,Age,wrong@email.com,abc,1000,1,Basic
7,Negative,Age,neg@email.com,-5,500,1,Standard
8,Missing,Email,,42,1200,1,Premium
9,Big,Points,big@email.com,38,999999999999,1,VIP
10,Normal,Person,normal@email.com,29,750,0,Basic
11,Extra,Fields,extra@email.com,33,900,1,Premium,EXTRA_COL,ANOTHER_EXTRA
12,Too,Few,toofew@email.com
13,Special,Chars,special@email.com,41,1100,1,Prëmîüm
14,Unicode,Name,unicode@email.com,36,1400,1,Standard
15,Null,Island,null@email.com,,,,
16,Good,Record1,good1@email.com,44,2100,1,VIP
17,Good,Record2,good2@email.com,27,300,0,Basic
18,Good,Record3,good3@email.com,51,1800,1,Premium
19,Float,Age,float@email.com,29.5,600,1,Standard
20,Huge,Age,huge@email.com,999,100,1,Basic"""

# Save to ADLS
file_path = source_path + "bad_customers.csv"
dbutils.fs.put(file_path, csv_data, overwrite=True)
print(f"✅ Bad data CSV saved to: {file_path}")
print(f"   Total rows: 20 (15 good, 5 intentionally bad)")

print("\n📋 Types of bad data included:")
print("   Row 6  → age is 'abc' (string in numeric field)")
print("   Row 7  → age is -5 (negative, logically invalid)")
print("   Row 8  → email is empty (missing required field)")
print("   Row 9  → loyalty_points overflow (huge number)")
print("   Row 11 → extra columns beyond schema")
print("   Row 12 → too few columns (missing fields)")
print("   Row 13 → special characters in segment")
print("   Row 15 → all optional fields are null")
print("   Row 19 → age is 29.5 (float in integer field)")
print("   Row 20 → age is 999 (logically invalid)")

In [0]:
customer_schema = '''
customer_id INT,
first_name STRING,
last_name STRING,
email STRING,
age INT,
loyalty_points INT,
is_active INT,
segment STRING
'''

# Reading with Auto Loader
df = spark.readStream\
    .schema(customer_schema)\
    .option("rescuedDataColumn", "_rescued_data")\
    .format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", schema_path)\
    .option("header", "true")\
    .load(source_path)

# Metadata columns
batch_id = uuid.uuid4().hex
df = (df
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_load_timestamp", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
    .withColumn("_batch_id", F.lit(batch_id))
)

# Writing stream
df.writeStream\
    .format("delta")\
    .option("checkpointLocation", checkpoint_path)\
    .trigger(availableNow=True)\
    .partitionBy("_load_date")\
    .start(target_path)\
    .awaitTermination()

In [0]:
df = spark.read.format("delta").load(target_path)
print(df.count())
print(df.filter('_rescued_data IS NOT NULL').count())
df.filter('_rescued_data IS NOT NULL').show(truncate=False)

# df.show()
# df.printSchema()


In [0]:
df = spark.read\
    .schema(customer_schema)\
    .option("mode", "FAILFAST")\
    .format("csv")\
    .option("header", "true")\
    .load(source_path)

df.show()

In [0]:
df_dropmalformed = spark.read\
    .schema(customer_schema)\
    .option("mode", "DROPMALFORMED")\
    .format("csv")\
    .option("header", "true")\
    .load(source_path)

df_dropmalformed.show()


In [0]:
bronze_df = spark.read.format("delta").load(target_path)

good_df = bronze_df.filter('_rescued_data IS NULL')
bad_df = bronze_df.filter('_rescued_data IS NOT NULL')

bad_df = (bad_df
    .withColumn("quarantine_reason", F.lit("_rescued_data is not null"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
)

bad_df.write.mode("overwrite").format("delta").save(quarantine_path)

print(bronze_df.count())
print(good_df.count())
print(bad_df.count())
bad_df.printSchema()

In [0]:
quarantine_df = spark.read.format("delta").load(quarantine_path)
quarantine_df.show(truncate=False)